# Low Weight Structure
Configuration: `stages/04_low_weight_structure/config/runtime.json`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
low_weight_structure_experiment_dir = ROOT / 'experiment/research/trapping_sets/stages/04_low_weight_structure'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

low_weight_structure_data_paths = ExperimentPaths(ROOT, low_weight_structure_experiment_dir / 'config/paths.json')

import json

low_weight_structure_config = json.loads((low_weight_structure_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.

from tqdm.auto import tqdm
from experiment.utils.trapping_sets import save_plotly

low_weight_structure_plots = low_weight_structure_data_paths.plots()
low_weight_structure_plots.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from codes.ca_code import CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from utils.graph_struct import build_tanner_graph_from_dem, err_subgraph_struct, det_subgraph_struct, local_incident_subgraph
from utils.graph_vis import visualize_graph_plotly, visualize_det_on_tanner_graph
from utils.misc import str2id as low_weight_structure_str2id, or_strs
from experiment.utils.trapping_sets import collect_low_weight_structure, low_weight_summary

low_weight_structure_code = CA_9_2_5()
low_weight_structure_data_dir = low_weight_structure_data_paths.run()
SUBGRAPH_DIR = low_weight_structure_data_paths.output('subgraph')
SUBGRAPH_DIR.mkdir(parents=True, exist_ok=True)
low_weight_structure_stats = low_weight_structure_data_paths.stats()
low_weight_structure_stats.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR = low_weight_structure_data_paths.input('error_weight')
low_weight_structure_types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}


### Subgraph

### Weight-2 population and structural features

Read the fixed-weight support table and extract error-induced and syndrome-induced graph features. Preserve row alignment when applying decoder-outcome masks to these feature tables.


In [ ]:
weight = 2
weight2_data = pd.read_csv(low_weight_structure_data_paths.input('weight_2_data_csv'), dtype=low_weight_structure_types)


In [ ]:
err_subgraph_analysis, det_subgraph_analysis = collect_low_weight_structure(low_weight_structure_code, weight2_data, weight=2, print_level=low_weight_structure_config['print_level'])
err_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_2_err_subgraph_csv'), index=False)
det_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_2_det_subgraph_csv'), index=False)

for population, features in tqdm([('error_induced', err_subgraph_analysis), ('detector_induced', det_subgraph_analysis)], desc='Trapping-set analysis', leave=True, disable=low_weight_structure_config['print_level'] == 1):
    low_weight_summary(weight2_data, features, weight=2, population=population).to_csv(low_weight_structure_data_paths.output('weight_2_population__summary_csv', population=population), index=False)


In [ ]:
err_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_2_err_subgraph_csv'))
det_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_2_det_subgraph_csv'))


In [ ]:
bfts_mask = (weight2_data['pred_b'] != weight2_data['true']) & (weight2_data['pred_t'] == weight2_data['true'])
bstf_mask = (weight2_data['pred_b'] == weight2_data['true']) & (weight2_data['pred_t'] != weight2_data['true'])
bf_mask = (weight2_data['pred_b'] != weight2_data['true']) & (weight2_data['pred_t'] != weight2_data['true'])
bs_mask = (weight2_data['pred_b'] == weight2_data['true']) & (weight2_data['pred_t'] == weight2_data['true'])


In [ ]:
low_weight_structure_counts = {
    'bfts': bfts_mask.sum(),
    'bstf': bstf_mask.sum(),
    'bf': bf_mask.sum(),
    'bs': bs_mask.sum(),
}

low_weight_structure_counts


#### Syndrome-induced Subgraph

### Compare syndrome-induced features

Contrast target disagreements with both-success controls in the graph of observed detectors. This representation does not identify the underlying physical error support uniquely.


In [ ]:
low_weight_structure_bfts_df = det_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bs_df = det_subgraph_analysis[bs_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bs_df['case_type'] = 'Both success'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bs_df], axis=0)


In [ ]:
from tqdm.auto import tqdm

features = compare_df.columns.difference(['WL_hash', 'case_type'])
low_weight_structure_fig, low_weight_structure_ax = plt.subplots(3, features.size // 3, figsize=(24, 12))
low_weight_structure_ax = low_weight_structure_ax.flatten()

for low_weight_structure_i, low_weight_structure_col in tqdm(enumerate(features), desc='Scientific sweep', disable=low_weight_structure_config['print_level'] == 1):
    sns.violinplot(
        data=compare_df,
        x='case_type',
        y=low_weight_structure_col,
        inner='box',
        cut=0,
        ax=low_weight_structure_ax[low_weight_structure_i],
    )
    low_weight_structure_ax[low_weight_structure_i].set_xlabel('')
    low_weight_structure_ax[low_weight_structure_i].set_ylabel(low_weight_structure_col)
    low_weight_structure_ax[low_weight_structure_i].set_title(f'Comparison of {low_weight_structure_col}')
    low_weight_structure_ax[low_weight_structure_i].tick_params(axis='x', rotation=15)

plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0093_1_png'), bbox_inches='tight')
plt.show()


#### Error-induced Subgraph

### Compare error-induced motifs

Compare the error-support graph features and WL-hash frequencies for the selected outcome groups. Truth-based support information is an offline explanatory variable.


In [ ]:
low_weight_structure_bfts_df = err_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bs_df = err_subgraph_analysis[bs_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bs_df['case_type'] = 'Both success'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bs_df], axis=0)


In [ ]:
wl_counts = compare_df.groupby(['case_type', 'WL_hash']).size().reset_index(name='count')
wl_counts['ratio'] = wl_counts['count'] / wl_counts.groupby('case_type')['count'].transform('sum')


In [ ]:
top_hashes = wl_counts.sort_values('count', ascending=False).groupby('case_type').head(10)['WL_hash'].unique()
plot_df = wl_counts[wl_counts['WL_hash'].isin(top_hashes)]
plt.figure(figsize=(12, 5))
sns.barplot(
    data=plot_df,
    x='WL_hash',
    y='ratio',
    hue='case_type',
)
plt.xlabel('WL_hash')
plt.ylabel('ratio within class')
plt.title('Most frequent local error-induced graph structures')
plt.xticks(rotation=90)
plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0098_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
err_subgraph_analysis[bfts_mask]


In [ ]:
weight2_data[bfts_mask]


In [ ]:
low_weight_structure_builder = CircuitBuilder(low_weight_structure_code)
low_weight_structure_circ = low_weight_structure_builder.get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1)
low_weight_structure_dem = low_weight_structure_circ.detector_error_model()
TG, low_weight_structure_ = build_tanner_graph_from_dem(low_weight_structure_dem)


##### WL_hash = 6fa6d6b6b4e35dfd93990869f812080a

### Inspect paired explanations of a syndrome

Display selected supports that share a detector pattern and compare their local neighborhoods. Their logical labels must be checked separately from the visual similarity of the graphs.


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '11010000000010', other_events={
    ('error', 'green', 'circle'): '010000010000000000000000000000000',
    ('bposd_corr', 'orange', 'circle'): '100000000000010000000000000000010',
}), low_weight_structure_data_paths.output('cell_0103_1_html'))


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '11010000000010', other_events={('error', 'green', 'circle'): '001000000100000000000000000000000'}), low_weight_structure_data_paths.output('cell_0104_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(local_incident_subgraph(TG, low_weight_structure_str2id(or_strs('010000010000000000000000000000000', '001000000100000000000000000000000'), low_weight_structure_dem.num_detectors))), low_weight_structure_data_paths.output('cell_0105_1_html'))


##### WL_hash = 8866aac84b5fa4e249b141d4d7076840

In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '11011000000110', other_events={('error', 'green', 'circle'): '010000000100000000000000000000000'}), low_weight_structure_data_paths.output('cell_0108_1_html'))


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '11011000000110', other_events={('error', 'green', 'circle'): '001000010000000000000000000000000'}), low_weight_structure_data_paths.output('cell_0109_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(local_incident_subgraph(TG, low_weight_structure_str2id(or_strs('010000000100000000000000000000000', '001000010000000000000000000000000'), low_weight_structure_dem.num_detectors))), low_weight_structure_data_paths.output('cell_0110_1_html'))


#### Error vs BPOSD Correction vs Tesseract Correction

### Subgraph

### Repeat at weight 3

Load the weight-3 cases, compute their structural features and form their outcome groups. Keep this population separate from the weight-2 comparisons.


In [ ]:
weight = 3
weight3_data = pd.read_csv(low_weight_structure_data_paths.input('weight_3_data_csv'), dtype=low_weight_structure_types)


In [ ]:
weight3_data['det'].value_counts()


In [ ]:
weight3_data[weight3_data['det'] == '00000000000000'][weight3_data[weight3_data['det'] == '00000000000000']['pred_b'] != weight3_data[weight3_data['det'] == '00000000000000']['true']]


In [ ]:
err_subgraph_analysis, det_subgraph_analysis = collect_low_weight_structure(low_weight_structure_code, weight3_data, weight=3, print_level=low_weight_structure_config['print_level'])
err_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_3_err_subgraph_csv'), index=False)
det_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_3_det_subgraph_csv'), index=False)

for population, features in tqdm([('error_induced', err_subgraph_analysis), ('detector_induced', det_subgraph_analysis)], desc='Trapping-set analysis', leave=True, disable=low_weight_structure_config['print_level'] == 1):
    low_weight_summary(weight3_data, features, weight=3, population=population).to_csv(low_weight_structure_data_paths.output('weight_3_population__summary_csv', population=population), index=False)


In [ ]:
err_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_3_err_subgraph_csv'))
det_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_3_det_subgraph_csv'))


In [ ]:
bfts_mask = (weight3_data['pred_b'] != weight3_data['true']) & (weight3_data['pred_t'] == weight3_data['true'])
bstf_mask = (weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])
bf_mask = (weight3_data['pred_b'] != weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])
bs_mask = (weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] == weight3_data['true'])


In [ ]:
low_weight_structure_counts = {
    'bfts': bfts_mask.sum(),
    'bstf': bstf_mask.sum(),
    'bf': bf_mask.sum(),
    'bs': bs_mask.sum(),
}

low_weight_structure_counts


#### Syndrome-induced Subgraph

### Compare weight-3 structure by outcome

Compare syndrome-induced features and error-induced motif frequencies in the two disagreement directions. Different selected groups require their own frequency denominators.


In [ ]:
low_weight_structure_bfts_df = det_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bstf_df = det_subgraph_analysis[bstf_mask].copy()
low_weight_structure_bs_df = det_subgraph_analysis[bs_mask].copy()
low_weight_structure_bf_df = det_subgraph_analysis[bf_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bstf_df['case_type'] = 'Tess fail / BPOSD success'
low_weight_structure_bs_df['case_type'] = 'Both success'
low_weight_structure_bf_df['case_type'] = 'Both fail'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bstf_df, low_weight_structure_bs_df, low_weight_structure_bf_df], axis=0)


In [ ]:
from tqdm.auto import tqdm

features = compare_df.columns.difference(['WL_hash', 'case_type'])
low_weight_structure_fig, low_weight_structure_ax = plt.subplots(3, features.size // 3, figsize=(24, 12))
low_weight_structure_ax = low_weight_structure_ax.flatten()

for low_weight_structure_i, low_weight_structure_col in tqdm(enumerate(features), desc='Scientific sweep', disable=low_weight_structure_config['print_level'] == 1):
    sns.violinplot(
        data=compare_df,
        x='case_type',
        y=low_weight_structure_col,
        inner='box',
        cut=0,
        ax=low_weight_structure_ax[low_weight_structure_i],
    )
    low_weight_structure_ax[low_weight_structure_i].set_xlabel('')
    low_weight_structure_ax[low_weight_structure_i].set_ylabel(low_weight_structure_col)
    low_weight_structure_ax[low_weight_structure_i].set_title(f'Comparison of {low_weight_structure_col}')
    low_weight_structure_ax[low_weight_structure_i].tick_params(axis='x', rotation=15)

plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0153_1_png'), bbox_inches='tight')
plt.show()


#### Error-induced Subgraph

In [ ]:
low_weight_structure_bfts_df = err_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bstf_df = err_subgraph_analysis[bstf_mask].copy()
low_weight_structure_bs_df = err_subgraph_analysis[bs_mask].copy()
low_weight_structure_bf_df = err_subgraph_analysis[bf_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bstf_df['case_type'] = 'Tess fail / BPOSD success'
low_weight_structure_bs_df['case_type'] = 'Both success'
low_weight_structure_bf_df['case_type'] = 'Both fail'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bstf_df, low_weight_structure_bs_df, low_weight_structure_bf_df], axis=0)


In [ ]:
wl_counts = compare_df.groupby(['case_type', 'WL_hash']).size().reset_index(name='count')
wl_counts['ratio'] = wl_counts['count'] / wl_counts.groupby('case_type')['count'].transform('sum')


In [ ]:
top_hashes = wl_counts.sort_values('count', ascending=False).groupby('case_type').head(10)['WL_hash'].unique()
plot_df = wl_counts[wl_counts['WL_hash'].isin(top_hashes)]
plt.figure(figsize=(12, 5))
sns.barplot(
    data=plot_df,
    x='WL_hash',
    y='ratio',
    hue='case_type',
)
plt.xlabel('WL_hash')
plt.ylabel('ratio within class')
plt.title('Most frequent local error-induced graph structures')
plt.xticks(rotation=90)
plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0158_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
wl_counts[wl_counts['case_type'] == 'BPOSD fail / Tess success']['WL_hash'].value_counts()


In [ ]:
err_subgraph_analysis[bfts_mask]


In [ ]:
weight3_data[bfts_mask]


In [ ]:
low_weight_structure_builder = CircuitBuilder(low_weight_structure_code)
low_weight_structure_circ = low_weight_structure_builder.get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1)
low_weight_structure_dem = low_weight_structure_circ.detector_error_model()
TG, low_weight_structure_ = build_tanner_graph_from_dem(low_weight_structure_dem)


##### WL_hash = f9115d16b0fc945295ac327c3b91ac61

### Inspect a selected weight-3 motif

Find the named hash group and visualize representative error supports and their combined neighborhood. The hash example is illustrative rather than an estimate of motif prevalence.


In [ ]:
err_subgraph_analysis[bfts_mask][err_subgraph_analysis[bfts_mask]['WL_hash'] == 'f9115d16b0fc945295ac327c3b91ac61']


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '10001010010000', other_events={('error', 'green', 'circle'): '100000100000000000010000000000000'}), low_weight_structure_data_paths.output('cell_0165_1_html'))


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '10001010010000', other_events={('error', 'green', 'circle'): '100000010000000000000010000000000'}), low_weight_structure_data_paths.output('cell_0166_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(local_incident_subgraph(TG, low_weight_structure_str2id(or_strs('100000100000000000010000000000000', '100000010000000000000010000000000'), low_weight_structure_dem.num_detectors))), low_weight_structure_data_paths.output('cell_0167_1_html'))


### Subgraph

### Repeat at weight 4

Construct the corresponding feature tables and outcome masks for weight-4 cases. Cross-weight differences must retain the weight-specific population and priors.


In [ ]:
weight = 4
weight4_data = pd.read_csv(low_weight_structure_data_paths.input('weight_4_data_csv'), dtype=low_weight_structure_types)


In [ ]:
weight4_data['det'].value_counts()


In [ ]:
weight4_data[weight4_data['det'] == '00000000000000']


In [ ]:
err_subgraph_analysis, det_subgraph_analysis = collect_low_weight_structure(low_weight_structure_code, weight4_data, weight=4, print_level=low_weight_structure_config['print_level'])
err_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_4_err_subgraph_csv'), index=False)
det_subgraph_analysis.to_csv(low_weight_structure_data_paths.output('weight_4_det_subgraph_csv'), index=False)

for population, features in tqdm([('error_induced', err_subgraph_analysis), ('detector_induced', det_subgraph_analysis)], desc='Trapping-set analysis', leave=True, disable=low_weight_structure_config['print_level'] == 1):
    low_weight_summary(weight4_data, features, weight=4, population=population).to_csv(low_weight_structure_data_paths.output('weight_4_population__summary_csv', population=population), index=False)


In [ ]:
err_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_4_err_subgraph_csv'))
det_subgraph_analysis = pd.read_csv(low_weight_structure_data_paths.output('weight_4_det_subgraph_csv'))


In [ ]:
bfts_mask = (weight4_data['pred_b'] != weight4_data['true']) & (weight4_data['pred_t'] == weight4_data['true'])
bstf_mask = (weight4_data['pred_b'] == weight4_data['true']) & (weight4_data['pred_t'] != weight4_data['true'])
bf_mask = (weight4_data['pred_b'] != weight4_data['true']) & (weight4_data['pred_t'] != weight4_data['true'])
bs_mask = (weight4_data['pred_b'] == weight4_data['true']) & (weight4_data['pred_t'] == weight4_data['true'])


In [ ]:
low_weight_structure_counts = {
    'bfts': bfts_mask.sum(),
    'bstf': bstf_mask.sum(),
    'bf': bf_mask.sum(),
    'bs': bs_mask.sum(),
}

low_weight_structure_counts


#### Syndrome-induced Subgraph

### Compare weight-4 features and motifs

Repeat the detector/error graph comparisons within the weight-4 groups. Structural enrichment motivates a candidate test; it does not on its own prove trapping behavior.


In [ ]:
low_weight_structure_bfts_df = det_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bstf_df = det_subgraph_analysis[bstf_mask].copy()
low_weight_structure_bs_df = det_subgraph_analysis[bs_mask].copy()
low_weight_structure_bf_df = det_subgraph_analysis[bf_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bstf_df['case_type'] = 'Tess fail / BPOSD success'
low_weight_structure_bs_df['case_type'] = 'Both success'
low_weight_structure_bf_df['case_type'] = 'Both fail'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bstf_df, low_weight_structure_bs_df, low_weight_structure_bf_df], axis=0)


In [ ]:
from tqdm.auto import tqdm

features = compare_df.columns.difference(['WL_hash', 'case_type'])
low_weight_structure_fig, low_weight_structure_ax = plt.subplots(3, features.size // 3, figsize=(24, 12))
low_weight_structure_ax = low_weight_structure_ax.flatten()

for low_weight_structure_i, low_weight_structure_col in tqdm(enumerate(features), desc='Scientific sweep', disable=low_weight_structure_config['print_level'] == 1):
    sns.violinplot(
        data=compare_df,
        x='case_type',
        y=low_weight_structure_col,
        inner='box',
        cut=0,
        ax=low_weight_structure_ax[low_weight_structure_i],
    )
    low_weight_structure_ax[low_weight_structure_i].set_xlabel('')
    low_weight_structure_ax[low_weight_structure_i].set_ylabel(low_weight_structure_col)
    low_weight_structure_ax[low_weight_structure_i].set_title(f'Comparison of {low_weight_structure_col}')
    low_weight_structure_ax[low_weight_structure_i].tick_params(axis='x', rotation=15)

plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0193_1_png'), bbox_inches='tight')
plt.show()


#### Error-induced Subgraph

In [ ]:
low_weight_structure_bfts_df = err_subgraph_analysis[bfts_mask].copy()
low_weight_structure_bstf_df = err_subgraph_analysis[bstf_mask].copy()
low_weight_structure_bs_df = err_subgraph_analysis[bs_mask].copy()
low_weight_structure_bf_df = err_subgraph_analysis[bf_mask].copy()
low_weight_structure_bfts_df['case_type'] = 'BPOSD fail / Tess success'
low_weight_structure_bstf_df['case_type'] = 'Tess fail / BPOSD success'
low_weight_structure_bs_df['case_type'] = 'Both success'
low_weight_structure_bf_df['case_type'] = 'Both fail'
compare_df = pd.concat([low_weight_structure_bfts_df, low_weight_structure_bstf_df, low_weight_structure_bs_df, low_weight_structure_bf_df], axis=0)


In [ ]:
wl_counts = compare_df.groupby(['case_type', 'WL_hash']).size().reset_index(name='count')
wl_counts['ratio'] = wl_counts['count'] / wl_counts.groupby('case_type')['count'].transform('sum')


In [ ]:
top_hashes = wl_counts.sort_values('count', ascending=False).groupby('case_type').head(10)['WL_hash'].unique()
plot_df = wl_counts[wl_counts['WL_hash'].isin(top_hashes)]
plt.figure(figsize=(12, 5))
sns.barplot(
    data=plot_df,
    x='WL_hash',
    y='ratio',
    hue='case_type',
)
plt.xlabel('WL_hash')
plt.ylabel('ratio within class')
plt.title('Most frequent local error-induced graph structures')
plt.xticks(rotation=90)
plt.tight_layout()

plt.savefig(low_weight_structure_data_paths.output('cell_0198_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
plot_df[plot_df['case_type'] == 'BPOSD fail / Tess success']['WL_hash'].value_counts()


In [ ]:
low_weight_structure_builder = CircuitBuilder(low_weight_structure_code)
low_weight_structure_circ = low_weight_structure_builder.get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1)
low_weight_structure_dem = low_weight_structure_circ.detector_error_model()
TG, low_weight_structure_ = build_tanner_graph_from_dem(low_weight_structure_dem)


##### WL_hash = a00babe068ceb1c17ad76a04d4c311d7

### Inspect a selected weight-4 example

Display the recorded support for the selected motif and compare its graph neighborhood. Read it together with the stored decoder outcomes and true logical label.


In [ ]:
err_subgraph_analysis[bfts_mask][err_subgraph_analysis[bfts_mask]['WL_hash'] == 'a00babe068ceb1c17ad76a04d4c311d7']


In [ ]:
weight4_data[bfts_mask].loc[3243]


In [ ]:
save_plotly(visualize_det_on_tanner_graph(TG, '11110110100110', other_events={('error', 'green', 'circle'): '100000000010010000000000000100000'}), low_weight_structure_data_paths.output('cell_0205_1_html'))


In [ ]:
save_plotly(visualize_graph_plotly(local_incident_subgraph(TG, low_weight_structure_str2id('100000000010010000000000000100000', low_weight_structure_dem.num_detectors))), low_weight_structure_data_paths.output('cell_0206_1_html'))
